### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import sqlite3

GROUP = 'gp01'
DATA_URLS = {'survey_points.csv': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp01/survey_points.csv'}
SCHEMA = {'survey_points.csv': {'kind': 'csv', 'fields': ['name', 'category', 'longitude', 'latitude', 'elevation_m', 'status']}}

def valid_cached(path, spec):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if spec["kind"] == "csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                return set(spec["fields"]).issubset(set(reader.fieldnames or []))
        if spec["kind"] == "geojson":
            document = json.loads(path.read_text(encoding="utf-8"))
            if document.get("type") != "FeatureCollection" or not isinstance(document.get("features"), list):
                return False
            return all(isinstance(feature, dict) and isinstance(feature.get("properties"), dict) and set(spec["fields"]).issubset(feature["properties"]) for feature in document["features"])
        if spec["kind"] == "gpkg":
            if path.read_bytes()[:16] != b"SQLite format 3\x00":
                return False
            with sqlite3.connect(path) as connection:
                if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                    return False
                layers = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents")}
            return set(spec["layers"]).issubset(layers)
    except Exception:
        return False
    return False

def valid_dataset_root(base):
    folder = base / GROUP
    return folder.is_dir() and all(valid_cached(folder / filename, spec) for filename, spec in SCHEMA.items())

candidates = (Path("downloads/python/data"), Path("../../data"), Path("data"))
DATA_ROOT = next((candidate for candidate in candidates if valid_dataset_root(candidate)), None)

if DATA_ROOT is None:
    DATA_ROOT = Path("data")
    target_root = DATA_ROOT / GROUP
    target_root.mkdir(parents=True, exist_ok=True)
    failures = []
    for filename, url in DATA_URLS.items():
        target = target_root / filename
        if valid_cached(target, SCHEMA[filename]):
            print("[OK] Already available:", target)
            continue
        if target.exists():
            target.unlink()
        partial = target.with_name(target.name + ".part")
        if partial.exists():
            partial.unlink()
        try:
            with urlopen(url, timeout=30) as response:
                payload = response.read()
            if not payload:
                raise RuntimeError("empty response")
            partial.write_bytes(payload)
            partial.replace(target)
            if not valid_cached(target, SCHEMA[filename]):
                raise RuntimeError("minimum validation failed")
            print("[OK] Downloaded and validated:", target)
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if target.exists():
                target.unlink()
            failures.append(filename + ": " + str(exc))
    if failures or not valid_dataset_root(DATA_ROOT):
        raise RuntimeError("Could not prepare dataset " + GROUP + ": " + "; ".join(failures))
    print("[OK] Dataset root:", DATA_ROOT.resolve())
else:
    print("[OK] Using validated dataset root:", (DATA_ROOT / GROUP).resolve())


# GP01 — First GeoPython Workflow

Load → Inspect → Understand → Filter → Map → Validate → Export

Σε κάθε στάδιο γράψε τι περιμένεις να αλλάξει και τι θα ελέγξεις. Το CSV είναι αρχικά DataFrame· μετά τη δημιουργία geometry θα γίνει GeoDataFrame. Μην προχωράς σε επόμενο στάδιο αν δεν μπορείς να ερμηνεύσεις το output του προηγούμενου.

## 1. Load — Φόρτωση δεδομένων

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path


In [ ]:
csv_path = DATA_ROOT / "gp01/survey_points.csv"
points = pd.read_csv(csv_path)
points.head()


## 2. Inspect — Πρώτη επιθεώρηση

Εμφάνισε το μέγεθος, τις στήλες, τους τύπους και τις κενές τιμές. Το `shape` είναι γραμμές/στήλες, τα `dtypes` επηρεάζουν τις πράξεις και το `isna().sum()` δείχνει πού χρειάζεται προσοχή.

In [ ]:
# Συμπλήρωσε τους ελέγχους
# print(points.shape)
# print(points.columns)
# print(points.dtypes)
# print(points.isna().sum())


## 3. Understand — Δημιουργία γεωμετρίας

In [ ]:
geometry = [Point(xy) for xy in zip(points["longitude"], points["latitude"]) ]
gdf = gpd.GeoDataFrame(points, geometry=geometry, crs="EPSG:4326")
gdf.head()


## 4. Filter — Επιλογή εγγραφών

In [ ]:
# Κράτησε checked σημεία με υψόμετρο πάνω από 70 m
selected = gdf[
    (gdf["elevation_m"] > 70) &
    (gdf["status"] == "checked")
] .copy()
selected


## 5. Map — Χάρτης

In [ ]:
ax = gdf.plot(figsize=(8, 6), color="lightgray")
selected.plot(ax=ax, color="red", markersize=60)


## 6. Validate — GeoPython QA checks

In [ ]:
# Συμπλήρωσε ή εκτέλεσε τους QA checks
# assert len(gdf) > 0
# assert gdf.geometry.notna().all()
# assert gdf.geometry.geom_type.eq("Point").all()
# assert gdf.crs is not None
# assert (selected["elevation_m"] > 70).all()


## 7. Export — Αποθήκευση αποτελέσματος

In [ ]:
output_dir = Path("outputs/gp01")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "high_checked_points.geojson"
# selected.to_file(output_path, driver="GeoJSON")


## Mini Challenge

Βρες τα `water` σημεία με υψόμετρο μικρότερο από 65 m, κάνε QA και αποθήκευσέ τα ως `water_low_points.geojson`.

In [ ]:
# Γράψε εδώ τη λύση σου
